# Notebook 2: Diagnóstico de Esquemas y Reconstrucción (Bronze)
**Autor:** Leonardo Figueroa, Miguel Flores y Steven Villegas

**Pipeline ETL - NYC Taxi Trip Records**

**Objetivo:** Diagnosticar esquemas de archivos fuente, homologar al esquema canónico y escribir la capa Bronze.


## Configuración Inicial


In [1]:
import sys, os, json, yaml, logging
from datetime import datetime
from pathlib import Path

# --- MEMORIA Y PYTHON ---
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
os.environ['PYSPARK_SUBMIT_ARGS'] = '--driver-memory 4g --executor-memory 4g pyspark-shell'
# -------------------------

sys.path.append(os.path.abspath('..'))
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger('etl')
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *

_config_path = os.path.join(os.path.abspath('..'), "config", "etl_config.yaml")
with open(_config_path, "r", encoding="utf-8") as _f:
    _cfg = yaml.safe_load(_f)
_spark_cfg = _cfg.get("spark", {})
builder = SparkSession.builder \
    .appName("NYC_Taxi_ETL_02") \
    .master(_spark_cfg.get("master", "local[*]"))
for k, v in _spark_cfg.get("config", {}).items():
    builder = builder.config(k, v)
spark = builder.getOrCreate()

from src.config_loader import *
from src.schema_recovery import diagnose_schema, build_bronze


---
## FASE 2: Diagnóstico y Reconstrucción de Esquema Canónico
**Objetivo:** Unificar los esquemas de los 3 servicios (yellow, green, fhvhv) bajo un esquema canónico común.


In [2]:
# Leer inventario desde la capa audit
inventory_path = os.path.join(METADATA_PATH, "audit_file_inventory")
if os.path.exists(inventory_path):
    df_inv = spark.read.parquet(inventory_path)
    inventory = [row.asDict() for row in df_inv.collect()]
    logger.info(f"Loaded {len(inventory)} inventory records from audit")
else:
    logger.warning("Inventory not found, using sample from raw directories")
    inventory = []
    for svc in SERVICES:
        sp = os.path.join(RAW_PATH, svc)
        if os.path.exists(sp):
            for y in YEARS:
                for m in get_months(y):
                    fp = os.path.join(sp, f"year={y}", f"month={m:02d}", f"{svc}_tripdata_{y}-{m:02d}.parquet")
                    if os.path.exists(fp):
                        try:
                            df = spark.read.parquet(fp)
                            import hashlib
                            inventory.append({
                                "service_type": svc, "file_name": f"{svc}_tripdata_{y}-{m:02d}.parquet",
                                "file_path": fp, "partition_year": y, "partition_month": m,
                                "read_status": "SUCCESS", "record_count": df.count(),
                                "column_count": len(df.columns),
                                "schema_hash": hashlib.md5(str(df.schema).encode()).hexdigest()[:16]
                            })
                        except:
                            pass
    logger.info(f"Built inventory from raw: {len(inventory)} files")


2026-06-18 17:48:15,570 - etl - INFO - Loaded 84 inventory records from audit


In [3]:
# Diagnóstico de esquemas
diagnosis = diagnose_schema(spark, inventory)
if diagnosis:
    df_diag = spark.createDataFrame(diagnosis)
    df_diag_pd = df_diag.toPandas()
    print("\nDiagnóstico de esquemas por archivo:")
    display(df_diag_pd)
    # Guardar diagnóstico
    diag_path = os.path.join(METADATA_PATH, "schema_diagnosis")
    df_diag.write.mode("overwrite").parquet(diag_path)
else:
    logger.warning("No diagnosis results")



Diagnóstico de esquemas por archivo:


,diagnosis_status,expected_columns,extra_columns,file_name,missing_columns,real_columns,service_type
0,MISSING_COLUMNS,19,"[cbd_congestion_fee, trip_type, ehail_fee]",green_tripdata_2025-10.parquet,[airport_fee],21,green
1,MISSING_COLUMNS,19,"[cbd_congestion_fee, trip_type, ehail_fee]",green_tripdata_2025-11.parquet,[airport_fee],21,green
2,MISSING_COLUMNS,19,"[cbd_congestion_fee, trip_type, ehail_fee]",green_tripdata_2025-12.parquet,[airport_fee],21,green
3,MISSING_COLUMNS,19,"[cbd_congestion_fee, trip_type, ehail_fee]",green_tripdata_2026-01.parquet,[airport_fee],21,green
4,MISSING_COLUMNS,19,"[cbd_congestion_fee, trip_type, ehail_fee]",green_tripdata_2026-02.parquet,[airport_fee],21,green
...,...,...,...,...,...,...,...
79,MISSING_COLUMNS,19,"[cbd_congestion_fee, trip_type, ehail_fee]",green_tripdata_2025-05.parquet,[airport_fee],21,green
80,MISSING_COLUMNS,19,"[cbd_congestion_fee, trip_type, ehail_fee]",green_tripdata_2025-06.parquet,[airport_fee],21,green
81,MISSING_COLUMNS,19,"[cbd_congestion_fee, trip_type, ehail_fee]",green_tripdata_2025-07.parquet,[airport_fee],21,green
82,MISSING_COLUMNS,19,"[cbd_congestion_fee, trip_type, ehail_fee]",green_tripdata_2025-08.parquet,[airport_fee],21,green


----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 51937)
Traceback (most recent call last):
  File "C:\Users\Usuario-PC\AppData\Local\Programs\Python\Python313\Lib\socketserver.py", line 318, in _handle_request_noblock
    self.process_request(request, client_address)
    ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Usuario-PC\AppData\Local\Programs\Python\Python313\Lib\socketserver.py", line 349, in process_request
    self.finish_request(request, client_address)
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Usuario-PC\AppData\Local\Programs\Python\Python313\Lib\socketserver.py", line 362, in finish_request
    self.RequestHandlerClass(request, client_address, self)
    ~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Usuario-PC\AppData\Local\Programs\Python\Python313\Lib\socketserver.py", line 766, in __init__
    self.handle()
    ~~~~~~~~~~~^^
  File "c:\Users\Usu

In [4]:
# Construcción de capa Bronze
bronze_records = build_bronze(spark, inventory)
if bronze_records:
    df_bronze_summary = spark.createDataFrame(bronze_records)
    print("\nResumen de carga Bronze:")
    display(df_bronze_summary.toPandas())
    logger.info(f"Bronze layer: {len(bronze_records)} partitions written")
else:
    logger.warning("No bronze records written")


2026-06-17 20:02:34,429 - src.schema_recovery - INFO - Bronze written: fhvhv_tripdata_2025-03.parquet
2026-06-17 20:02:41,412 - src.schema_recovery - INFO - Bronze written: fhvhv_tripdata_2025-04.parquet
2026-06-17 20:02:48,283 - src.schema_recovery - INFO - Bronze written: fhvhv_tripdata_2025-05.parquet
2026-06-17 20:02:55,016 - src.schema_recovery - INFO - Bronze written: fhvhv_tripdata_2025-06.parquet
2026-06-17 20:03:01,743 - src.schema_recovery - INFO - Bronze written: fhvhv_tripdata_2025-07.parquet
2026-06-17 20:03:08,586 - src.schema_recovery - INFO - Bronze written: fhvhv_tripdata_2025-08.parquet
2026-06-17 20:03:15,083 - src.schema_recovery - INFO - Bronze written: fhvhv_tripdata_2025-09.parquet
2026-06-17 20:03:21,649 - src.schema_recovery - INFO - Bronze written: fhvhv_tripdata_2025-10.parquet
2026-06-17 20:03:28,210 - src.schema_recovery - INFO - Bronze written: fhvhv_tripdata_2025-11.parquet
2026-06-17 20:03:35,268 - src.schema_recovery - INFO - Bronze written: fhvhv_tripd


Resumen de carga Bronze:


,file_name,month,records,service_type,status,year
0,fhvhv_tripdata_2025-03.parquet,3,20536879,fhvhv,BRONZE_WRITTEN,2025
1,fhvhv_tripdata_2025-04.parquet,4,19753983,fhvhv,BRONZE_WRITTEN,2025
2,fhvhv_tripdata_2025-05.parquet,5,21091193,fhvhv,BRONZE_WRITTEN,2025
3,fhvhv_tripdata_2025-06.parquet,6,19868009,fhvhv,BRONZE_WRITTEN,2025
4,fhvhv_tripdata_2025-07.parquet,7,19653012,fhvhv,BRONZE_WRITTEN,2025
...,...,...,...,...,...,...
79,green_tripdata_2025-05.parquet,5,55399,green,BRONZE_WRITTEN,2025
80,green_tripdata_2025-06.parquet,6,49390,green,BRONZE_WRITTEN,2025
81,green_tripdata_2025-07.parquet,7,48205,green,BRONZE_WRITTEN,2025
82,green_tripdata_2025-08.parquet,8,46306,green,BRONZE_WRITTEN,2025


2026-06-17 20:07:22,672 - etl - INFO - Bronze layer: 84 partitions written


---
## Resumen de Diagnóstico y Bronze
Los archivos se han transformado al esquema canónico y escrito en data/bronze/.
